In [ ]:
!pip install diffusers transformers torch matplotlib -q !pip install transformers sentence-transformers pandas -q import torch from transformers import pipeline, set_seed from sentence_transformers import SentenceTransformer, util import pandas as pd set_seed(42) device = 0 if torch.cuda.is_available() else -1 generator = pipeline(     "text-generation",     model="gpt2",     device=device
)
print("Model loaded successfully!\n") base_text = "Artificial Intelligence is transforming industries by automating complex tasks and improving decision making." styles = {     "Formal":
        f"Rewrite the following in a formal academic tone:\n{base_text}",     "Informal":
        f"Rewrite this in a friendly casual tone:\n{base_text}",     "Technical":
        f"Explain this in a technical way with precise terminology:\n{base_text}",     "Poetic":
        f"Convert this into a poetic form:\n{base_text}",     "Humorous":
        f"Rewrite this in a funny humorous way:\n{base_text}",     "Persuasive":
        f"Rewrite this in a persuasive way to convince someone:\n{base_text}" }
 results = []
print("=== STYLE GENERATED OUTPUTS ===\n") for style, prompt in styles.items():
    output = generator(         prompt,         max_new_tokens=100,         temperature=0.85,         top_p=0.95,         num_return_sequences=2
    )
    for i, o in enumerate(output):         text = o['generated_text']         print(f"\n--- {style} Style | Version {i+1} ---")         print(text)         print("-"*80)         results.append([style, i+1, text]) model = SentenceTransformer('all-MiniLM-L6-v2') base_embedding = model.encode(base_text, convert_to_tensor=True) analysis = []
print("\n\n=== SEMANTIC SIMILARITY ANALYSIS ===\n") for style, version, text in results:
    emb = model.encode(text, convert_to_tensor=True)     similarity = util.cos_sim(base_embedding, emb).item()     print(f"{style} (v{version}) → Similarity: {similarity:.4f}")     analysis.append([style, version, similarity]) length_analysis = [] for style, version, text in results:
    length = len(text.split())     length_analysis.append([style, version, length])
df_similarity = pd.DataFrame(analysis, columns=["Style", "Version", "Semantic Similarity"]) df_length = pd.DataFrame(length_analysis, columns=["Style", "Version", "Word Count"])

print("\n=== SIMILARITY TABLE ===\n") print(df_similarity)
print("\n=== LENGTH TABLE ===\n") print(df_length) best = df_similarity.sort_values(by="Semantic Similarity", ascending=False).iloc[0] print("\nBest Style Based on Meaning Preservation:") print(best)

